# 08 · 판독과 분만 결과

**질문 1:** 파형 특징에 임상정보를 더하면 구간별 전문가 판독 예측이 좋아지는가?

**질문 2:** 실제 전문가 판독의 기록 요약을 더하면 관측된 분만 결과 예측이 좋아지는가?

| 분석 | 한 행 | 예측 표적 | 비교 |
|---|---|---|---|
| 임상정보의 증분 | 5분 구간 | 전문가 이상 판독 | Cat28 + 임상정보 − Cat28 |
| 판독 정보의 증분 | 기록 | pH·APGAR 기준의 이진 결과 | 임상정보 + 실제 판독 요약 − 임상정보 |

두 분석의 표적을 구분한다. 분만 결과나 Emergency를 첫 번째 모델의 입력으로 쓰지 않는다. 두 번째 분석에는 모델의 판독 예측 점수가 아니라 **실제 전문가 판독 요약**을 사용한다.

## 1. 입력과 설정

00의 `data/prepared/records.csv`, 01의 `data/results/protocol.json`, 02의 `features.csv`, 04의 Cat28 예측을 읽는다. 결과는 `data/results/`에 저장한다.

첫 분석은 기존 산모 train/val/test를 유지한다. 분만 결과 분석은 관측된 기록 전체에서 새 산모 5-fold OOF를 만들며, 기존 test와 독립된 확증 연구가 아니다. 학습 예산은 04와 동일하고 bootstrap은 2,000회다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, train_test_split

PREPARED = Path("data/prepared")
RESULTS = Path("data/results")
SEEDS = (42, 43, 44)
CV_FOLDS = 5
TREE_ITERATIONS = 1000
TREE_PATIENCE = 80
THREADS = 2
N_BOOTSTRAP = 2000
BOOTSTRAP_SEED = 42
ID_TYPES = {key: str for key in ("row_id", "record_id", "mother_id", "site")}

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

In [ ]:
# Preserve the Cat28 feature order used in 02 and 04.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s", "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel", "n_severe_decel", "n_prolonged_decel",
    "hist_width", "hist_median", "hist_mode",
]
# Only these 13 clinical inputs are allowed; outcomes are not candidate features.
CLINICAL = [
    "height", "weight", "gravida", "para", "sbp", "dbp", "ghtn",
    "hypertension", "gdm", "dm", "preeclampsia", "maternal_age", "gestational_age",
]
READING = ["abnormal_fraction", "longest_abnormal_run"]
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target", "split",
    "record_all_normal", "signal_index", "score", "threshold",
]
METRIC_COLUMNS = ["metric", "point", "low", "high", "valid_bootstrap", "mothers", "segments"]
OUTCOME_PREDICTION_COLUMNS = ["row_id", "record_id", "mother_id", "site", "target", "score", "threshold", "fold"]

In [ ]:
issues = []
protocol = {}
protocol_error = None
try:
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
    split_seed = int(protocol["split_seed"])
except (OSError, ValueError, KeyError, TypeError) as error:
    protocol_error = f"{type(error).__name__}: {error}"
    issues.append(dict(stage="input", job="protocol", reason=protocol_error))
    print('분할 설정을 읽을 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
records = pd.DataFrame()
records_error = None
try:
    # Example: record_id "DEMO854050_0" stays a string; numeric missing values stay NaN.
    records = pd.read_csv(PREPARED / "records.csv", dtype=ID_TYPES)
    if records.empty or records["record_id"].duplicated().any() or records["row_id"].duplicated().any():
        raise ValueError("기록 표가 비었거나 기록 ID가 중복됩니다")
    if records[list(ID_TYPES)].isna().any().any():
        raise ValueError("기록 ID·산모 ID·사이트에 결측이 있습니다")
except (OSError, ValueError, KeyError, TypeError) as error:
    records_error = f"{type(error).__name__}: {error}"
    issues.append(dict(stage="input", job="records", reason=records_error))
    print('기록 표를 이용할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
if records_error is None:
    print("record table shape =", records.shape)
    print("mothers =", records["mother_id"].nunique())
    # Example: one record has 4 original labels, 3 retained windows, abnormal_fraction=0.5.
    # The reading summary still describes all 4 original labels, not just the retained 3.
    preview = ["record_id", "mother_id", "n_segments", "n_kept"] + READING

## 2. 행을 맞추고 같은 기준으로 평가한다

두 모델의 같은 구간 또는 기록을 `row_id`로 정렬한다. 일부 행의 교집합으로 평가 대상을 바꾸지 않는다. AP·AUROC·Brier를 계산하며, 차이는 항상 **정보를 추가한 모델 − 기준 모델**이다. AP·AUROC는 양수, Brier는 음수일 때 추가 모델에 유리하다.

산모를 복원추출할 때 두 모델에 같은 횟수를 적용한다. 표의 `segments` 열 이름은 원본 형식을 유지한다. 첫 분석에서는 구간 수, 분만 결과 분석에서는 기록 수를 뜻한다.

In [ ]:
def align_rows(left, right):
    # Example: left IDs [A, B], right IDs [B, A] -> reorder right to [A, B].
    # Missing B is an error; comparing only the intersection changes the cohort.
    if left["row_id"].duplicated().any() or right["row_id"].duplicated().any():
        raise ValueError("평가 행 ID가 중복됩니다")
    if set(left["row_id"]) != set(right["row_id"]):
        raise ValueError("비교할 평가 행 집합이 다릅니다")
    # Reordering rows never changes their labels, records, or mother IDs.
    aligned = right.set_index("row_id").loc[left["row_id"]].reset_index()
    for column in ("mother_id", "target", "record_id"):
        if not np.array_equal(left[column].to_numpy(), aligned[column].to_numpy()):
            raise ValueError("평가 행 정보 불일치: " + column)
    return aligned

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B] -> mother_index [0, 0, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")
    # Example: sorted scores [0.9, 0.9, 0.3] -> group ends [1, 2].
    # Tied scores are one threshold, independent of their row order.
    ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, ends=ends,
    )

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: every mother appears once, so every row has weight 1.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> weights [2, 2, 0] for [A, A, B].
    weights = counts[data["mother_index"]]
    labels = data["labels"]
    # Denominators count weighted positive and negative rows.
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # Skip a resample containing only negative or only positive rows.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["ends"]
    # Count rows at or above each distinct score threshold.
    # Example: distinct descending scores with labels [1, 0, 1] -> TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = (1 + 2/3)/2 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # Those labels with distinct scores give AUROC=3/4.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label 1, score 0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (data["scores"] - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
def resample_metrics(left_data, right_data, n_boot, seed):
    samples = {metric: [] for metric in ("AP", "AUROC", "Brier")}
    mother_count = len(left_data["mothers"])
    rng = np.random.default_rng(seed)
    for _ in range(n_boot):
        # Example: mothers [A, A, C] -> counts [2, 0, 1] for [A, B, C].
        # All of A's rows get weight 2 on both sides of the comparison.
        counts = np.bincount(rng.integers(mother_count, size=mother_count), minlength=mother_count)
        left = weighted_metrics(left_data, counts)
        right = weighted_metrics(right_data, counts) if right_data is not None else None
        if left is None or (right_data is not None and right is None):
            continue
        for metric in samples:
            value = left[metric]
            if right is not None:
                value -= right[metric]
            if np.isfinite(value):
                samples[metric].append(value)
    return samples

In [ ]:
def metric_intervals(left, right=None, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    left_data = prepare_metrics(left)
    right_data = prepare_metrics(align_rows(left, right)) if right is not None else None
    point = weighted_metrics(left_data)
    reference = weighted_metrics(right_data) if right_data is not None else None
    if point is None or (right_data is not None and reference is None):
        raise ValueError("평가에는 양성·음성 행이 모두 필요합니다")
    if reference is not None:
        # Example: added AP 0.65 - baseline AP 0.60 -> +0.05.
        point = {metric: value - reference[metric] for metric, value in point.items()}
    samples = resample_metrics(left_data, right_data, n_boot, seed)
    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Central 95% of valid draws; one-class draws do not enter the interval.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            # n_boot=0 -> point estimate only, NaN endpoints, valid_bootstrap=0.
            valid_bootstrap=len(samples[metric]),
            mothers=len(left_data["mothers"]), segments=len(left),
        ))
    return pd.DataFrame(rows)

In [ ]:
# Perfectly ordered scores produce AP=1; a reversed order does not.
example_base = pd.DataFrame({
    "row_id": ["R1", "R2"], "record_id": ["R1", "R2"],
    "mother_id": ["M1", "M2"], "target": [0, 1], "score": [0.8, 0.2],
})
example_added = example_base.assign(score=[0.2, 0.8]).iloc[::-1]
# IDs restore the row order before subtraction: AP difference = 1 - 0.5 = 0.5.
# Brier difference = 0.04 - 0.64 = -0.60; no bootstrap is needed for this example.
display(metric_intervals(example_added, example_base, n_boot=0))

## 3. 학습과 예측의 공통 정의

CatBoost는 훈련 집합으로 학습하고 검증 집합으로 조기 종료한다. 임상정보의 수치 결측은 NaN으로 유지하여 CatBoost가 처리한다. 분만 결과의 결측을 정상 표적으로 채우는 것은 별개이며, 그렇게 하지 않는다.

In [ ]:
def fit_catboost(train, tune, columns, seed):
    from catboost import CatBoostClassifier

    if any(set(part["target"].unique()) != {0, 1} for part in (train, tune)):
        raise ValueError("학습·검증에 양성·음성 라벨이 모두 필요합니다")
    # Every arm uses the same CatBoost settings as 04; inputs are specified below.
    model = CatBoostClassifier(
        iterations=TREE_ITERATIONS, depth=6, learning_rate=0.05,
        loss_function="Logloss", eval_metric="PRAUC", auto_class_weights="SqrtBalanced",
        random_seed=seed, thread_count=THREADS,
        allow_writing_files=False, verbose=False,
    )
    # Only tune controls early stopping; test is never passed to fit().
    model.fit(
        train[columns], train["target"],
        eval_set=(tune[columns], tune["target"]),
        # Example: 80 unimproved rounds can stop before the 1,000-round cap.
        early_stopping_rounds=TREE_PATIENCE,
    )
    return model

In [ ]:
def make_prediction(frame, scores):
    # Window rows carry split/signal_index; record rows do not have those columns.
    columns = [column for column in PREDICTION_COLUMNS[:-2] if column in frame]
    prediction = frame[columns].copy()
    # scores[i] belongs to frame.iloc[i], before any ID alignment for comparison.
    prediction["score"] = np.asarray(scores, dtype=float)
    prediction["threshold"] = np.nan  # This experiment does not set alarm thresholds.
    if prediction["row_id"].duplicated().any() or not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 ID가 중복되거나 점수가 유한하지 않습니다")
    return prediction

In [ ]:
def plot_difference(table, metric, title):
    selected = table.loc[table["metric"].eq(metric)]
    if selected.empty:
        print("그릴 비교 결과가 없습니다.")
        return
    positions = np.arange(len(selected))
    figure, axis = plt.subplots(figsize=(8, max(3, len(selected) * 0.4)))
    # Each dot is added-minus-baseline; horizontal lines are mother-bootstrap CIs.
    axis.hlines(positions, selected["low"], selected["high"], color="C0")
    axis.scatter(selected["point"], positions)
    axis.axvline(0, color="gray", linestyle="--")
    axis.set_yticks(positions, selected["comparison"])
    axis.set_xlabel(metric + " difference (added minus baseline)")
    axis.set_title(title)
    figure.tight_layout()
    plt.show()

## 4. 같은 구간에 임상정보를 연결한다

한 기록의 임상정보를 그 기록의 여러 구간에 붙인다. 예를 들어 R1의 구간이 3개면 같은 산모 정보가 세 행에 들어간다. 연결 전후 구간 수를 유지하며, 01의 기존 분할을 그대로 읽는다.

In [ ]:
emr_data = pd.DataFrame()
emr_input_error = None
try:
    if records_error or protocol_error:
        raise ValueError(records_error or protocol_error)
    if not protocol.get("split_ready"):
        raise ValueError("01의 산모 분할이 완료되지 않았습니다")
    features = pd.read_csv(RESULTS / "features.csv", dtype=ID_TYPES)
    # many_to_one: many windows may use one record, but record_id must be unique.
    emr_data = features.merge(
        records[["record_id"] + CLINICAL], on="record_id",
        how="left", validate="many_to_one", indicator=True,
    )
    if not emr_data["_merge"].eq("both").all():
        raise ValueError("기록 표에 연결되지 않은 구간이 있습니다")
    emr_data = emr_data.drop(columns="_merge")
    baseline_prediction = pd.read_csv(RESULTS / "pred_cat28.csv", dtype=ID_TYPES)
except (OSError, ValueError, KeyError, TypeError) as error:
    emr_input_error = f"{type(error).__name__}: {error}"
    issues.append(dict(stage="emr_input", job="Cat28 + clinical", reason=emr_input_error))
    print('임상정보 비교를 준비할 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
def split_holdout(frame):
    parts = []
    for split in ("train", "val", "test"):
        # This reads saved assignments; it does not draw a fresh split per arm.
        part = frame.loc[frame["split"].eq(split)].copy()
        # Example: [0, 0, 0] alone cannot support the planned binary comparison.
        if set(part["target"].unique()) != {0, 1}:
            raise ValueError("train/val/test 각각 정상·비정상 라벨이 필요합니다")
        parts.append(part)
    return parts

In [ ]:
def check_reference(pool, prediction):
    # Require the same evaluation rows and original validation/test assignments.
    aligned = align_rows(pool, prediction)
    if not np.array_equal(pool["split"], aligned["split"]):
        raise ValueError("Cat28 기준 예측의 분할이 현재 입력과 다릅니다")
    if not np.isfinite(aligned["score"]).all():
        raise ValueError("Cat28 기준 예측에 유한하지 않은 점수가 있습니다")
    return aligned

In [ ]:
if emr_input_error is None:
    try:
        train, validation, test = split_holdout(emr_data)
        evaluation_pool = pd.concat([validation, test])
        baseline_prediction = check_reference(evaluation_pool, baseline_prediction)
        print("input columns =", len(CAT28), "+", len(CLINICAL), "=", len(CAT28 + CLINICAL))
        print("train / validation / test windows =", len(train), len(validation), len(test))
    except (ValueError, KeyError) as error:
        emr_input_error = str(error)
        issues.append(dict(stage="emr_input", job="split/reference", reason=emr_input_error))
        print('분할·기준 예측을 이용할 수 없습니다:', '[details omitted from notebook output]')

## 5. 세 seed를 학습하고 검증 AP로 고른다

42·43·44를 모두 학습한 뒤 검증 AP로 하나를 고른다. 동점은 작은 seed다. 시험 집합으로 seed를 선택하지 않는다. 같은 seed끼리의 시험 차이는 점 추정만 계산하고, 주 비교는 각 모델에서 선택한 seed끼리 산모 bootstrap을 한다.

seed별 기준 예측이 없으면 그 seed의 보조 비교만 미산출로 남긴다. 추가 모델의 학습 성공 여부와 보조 비교 성공 여부는 따로 기록한다.

In [ ]:
def fit_emr_candidate(train, validation, test, seed):
    columns = CAT28 + CLINICAL
    model = fit_catboost(train, validation, columns, seed)
    pool = pd.concat([validation, test])
    prediction = make_prediction(pool, model.predict_proba(pool[columns])[:, 1])
    validation_rows = prediction.loc[prediction["split"].eq("val")]
    val_ap = average_precision_score(validation_rows["target"], validation_rows["score"])
    # Return all val/test rows, but only validation AP can select the seed.
    return prediction, float(val_ap)

In [ ]:
emr_predictions = {}
emr_candidates = []
if emr_input_error is None:
    for seed in SEEDS:
        try:
            prediction, val_ap = fit_emr_candidate(train, validation, test, seed)
            emr_predictions[seed] = prediction
            emr_candidates.append(dict(seed=seed, val_AP=val_ap))
            print("EMR seed", seed, "complete", flush=True)
        except Exception as error:
            issues.append(dict(stage="emr_fit", job=str(seed), reason=str(error)))
            print('EMR seed', seed, 'unavailable:', '[details omitted from notebook output]', flush=True)
selection = pd.DataFrame(emr_candidates, columns=["seed", "val_AP"])
display(selection)

In [ ]:
same_seed_parts = []
for seed, prediction in emr_predictions.items():
    try:
        baseline = pd.read_csv(RESULTS / "seeds" / f"pred_cat28_{seed}.csv", dtype=ID_TYPES)
        baseline = check_reference(evaluation_pool, baseline)
        added_test = prediction.loc[prediction["split"].eq("test")]
        baseline_test = baseline.loc[baseline["split"].eq("test")]
        # Example: seed 42 is compared with Cat28 seed 42, without a CI.
        difference = metric_intervals(added_test, baseline_test, n_boot=0)
        same_seed_parts.append(difference.assign(seed=seed, comparison="EMR + Cat28 - Cat28"))
    except Exception as error:
        issues.append(dict(stage="emr_same_seed", job=str(seed), reason=str(error)))

In [ ]:
emr_paired = pd.DataFrame(columns=METRIC_COLUMNS + ["comparison", "seed"])
emr_same_seed = pd.DataFrame(columns=METRIC_COLUMNS + ["seed", "comparison"])
if same_seed_parts:
    emr_same_seed = pd.concat(same_seed_parts, ignore_index=True)
selected_emr_seed = None
if len(emr_predictions) == len(SEEDS):
    try:
        # Example: AP [0.70, 0.72, 0.72] at seeds [42, 43, 44] -> seed 43.
        best = max(emr_candidates, key=lambda row: (row["val_AP"], -row["seed"]))
        selected_emr_seed = best["seed"]
        prediction = emr_predictions[selected_emr_seed]
        difference = metric_intervals(
            prediction.loc[prediction["split"].eq("test")],
            baseline_prediction.loc[baseline_prediction["split"].eq("test")],
        )
        emr_paired = difference.assign(comparison="EMR + Cat28 - Cat28", seed=selected_emr_seed)
    except Exception as error:
        issues.append(dict(stage="emr_paired", job="selected seeds", reason=str(error)))
print("selected EMR seed =", selected_emr_seed)
display(emr_paired)
plot_difference(emr_paired, "AP", "Expert reading: added clinical information")

## 6. 분만 결과의 관측 범위와 표적을 만든다

분만 결과는 기록당 한 값이다. 아래 범위 안의 값만 관측으로 취급하고, 범위 밖 값과 결측은 해당 결과 분석에서 제외한다. 경계값과 같은 경우는 음성이다. 이 기준은 원본 실험의 정의이며 여기서 새로 선택하지 않는다.

| 결과 | 관측 범위 (양 끝 포함) | 양성 표적 |
|---|---|---|
| UA.pH | 6–8 | pH < 7.2 |
| APGAR1 | 0–10 | 1분 APGAR < 7 |
| APGAR5 | 0–10 | 5분 APGAR < 7 |

`abnormal_fraction`과 `longest_abnormal_run`은 00이 만든 원래 전체 판독 요약을 사용한다. 예를 들어 연속된 네 구간의 라벨이 `[0, 1, 1, 0]`이면 이상 비율은 0.5, 가장 긴 이상 연속 길이는 2구간이다. 특징 추출 후 남은 구간만으로 다시 계산하지 않는다.

In [ ]:
OUTCOMES = [
    ("UA.pH", "ua_ph", 6, 8, 7.2),
    ("APGAR1", "apgar1", 0, 10, 7),
    ("APGAR5", "apgar5", 0, 10, 7),
]
COHORT_COLUMNS = ["outcome", "cohort", "records", "mothers", "reading_positive_fraction"]
ASSOCIATION_COLUMNS = ["outcome", "records", "positives", "prevalence", "reading_AP", "reading_AUROC"]

# Names and cutoffs are fixed before examining association or OOF results.
print("outcome definitions =", OUTCOMES)

In [ ]:
def observed_outcome(records, field, low, high, cutoff):
    # Example: pH [7.1, 7.2, 9.0, NaN] -> observed [True, True, False, False].
    observed = records[field].between(low, high)
    valid = records.loc[observed].copy()
    # Only observed rows receive a target: pH 7.1 -> 1, pH 7.2 -> 0.
    # Missing outcomes never become target=0 through a comparison with NaN.
    valid["target"] = (valid[field] < cutoff).astype(int)
    return observed, valid

In [ ]:
example_records = pd.DataFrame({
    "record_id": ["R1", "R2", "R3", "R4"],
    "ua_ph": [7.1, 7.2, 9.0, np.nan],
})
example_observed, example_valid = observed_outcome(example_records, "ua_ph", 6, 8, 7.2)
print("observed mask =", example_observed.tolist())
print("excluded record IDs =", example_records.loc[~example_observed, "record_id"].tolist())
display(example_valid[["record_id", "ua_ph", "target"]])

In [ ]:
def outcome_cohorts(records, observed, name):
    rows = []
    for group, mask in (("observed", observed), ("missing_or_invalid", ~observed)):
        subset = records.loc[mask]
        # Example: 3 positive-reading records out of 10 -> fraction 0.3.
        # This is the fraction of records with ANY abnormal reading, not window prevalence.
        rows.append(dict(
            outcome=name, cohort=group, records=len(subset),
            mothers=subset["mother_id"].nunique(),
            reading_positive_fraction=subset["any_abnormal"].mean(),
        ))
    return rows

In [ ]:
def reading_association(valid, name):
    if set(valid["target"].unique()) != {0, 1}:
        return None
    # Use the actual abnormal-window fraction as a ranking score for the outcome.
    # Example: 2 abnormal labels out of 4 -> score 0.5, not an outcome probability.
    labels = valid["target"]
    reading_score = valid["abnormal_fraction"]
    return dict(
        outcome=name, records=len(valid), positives=int(labels.sum()),
        prevalence=labels.mean(),
        reading_AP=average_precision_score(labels, reading_score),
        reading_AUROC=roc_auc_score(labels, reading_score),
    )

In [ ]:
outcome_data = {}
cohort_rows, association_rows = [], []
if records_error is None:
    for name, field, low, high, cutoff in OUTCOMES:
        try:
            observed, valid = observed_outcome(records, field, low, high, cutoff)
            outcome_data[name] = valid
            cohort_rows.extend(outcome_cohorts(records, observed, name))
            association = reading_association(valid, name)
            if association is not None:
                association_rows.append(association)
        except Exception as error:
            issues.append(dict(stage="outcome_description", job=name, reason=str(error)))
cohorts = pd.DataFrame(cohort_rows, columns=COHORT_COLUMNS)
associations = pd.DataFrame(association_rows, columns=ASSOCIATION_COLUMNS)
display(cohorts)
display(associations)

In [ ]:
for name, valid in outcome_data.items():
    if "any_abnormal" not in valid:
        continue
    # Rows: any abnormal expert reading (0/1); columns: observed outcome target (0/1).
    # Each cell counts records, so twins remain separate records here.
    print(name, "expert reading x outcome (records)")
    display(pd.crosstab(valid["any_abnormal"], valid["target"]))

## 7. 산모 5-fold OOF로 분만 결과를 예측한다

관측된 기록을 산모 단위로 외부 5-fold에 나눈다. 매 fold의 평가 산모는 학습과 조기 종료에서 모두 제외한다. 나머지 산모 중 20%를 내부 validation으로 사용하며, 두 모델이 같은 내부 분할을 공유한다.

기준 모델은 임상 변수 13개, 추가 모델은 여기에 실제 판독 요약 2개를 더한 15개를 사용한다. 모델 seed는 원본처럼 **42 하나로 고정**한다. 내부 분할 seed는 `split_seed + fold`이고 fold 번호는 0부터 시작한다.

In [ ]:
def split_inner_mothers(pool, seed):
    # Example: twins with targets [0, 1] -> mother target 1 for inner stratification.
    mother_labels = pool.groupby("mother_id")["target"].max()
    train_ids, validation_ids = train_test_split(
        mother_labels.index.to_numpy(), test_size=0.2,
        stratify=mother_labels.to_numpy(), random_state=seed,
    )
    # 100 mothers -> 80 training and 20 validation mothers, not necessarily 80:20 rows.
    train = pool.loc[pool["mother_id"].isin(train_ids)]
    validation = pool.loc[pool["mother_id"].isin(validation_ids)]
    return train, validation

In [ ]:
def outcome_folds(valid, split_seed):
    # The outer splitter stratifies record targets while grouping all rows of a mother.
    splitter = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=split_seed)
    for fold, (pool_indices, held_indices) in enumerate(
        splitter.split(valid, valid["target"], valid["mother_id"])
    ):
        pool = valid.iloc[pool_indices].copy()
        held = valid.iloc[held_indices].copy()
        train, validation = split_inner_mothers(pool, split_seed + fold)
        # Example: fold 0 uses inner seed 42; fold 1 uses 43 when split_seed=42.
        yield fold, train, validation, held

In [ ]:
if protocol_error is None and outcome_data:
    sample_name = next(iter(outcome_data))
    try:
        fold, sample_train, sample_validation, sample_held = next(
            outcome_folds(outcome_data[sample_name], split_seed)
        )
        print("example outcome / fold =", sample_name, fold)
        for name, subset in (("train", sample_train), ("validation", sample_validation), ("held", sample_held)):
            print(name, "records / mothers =", len(subset), subset["mother_id"].nunique())
        source_mothers = set(sample_train["mother_id"]) | set(sample_validation["mother_id"])
        print("held/source mother overlap =", len(set(sample_held["mother_id"]) & source_mothers))
    except (ValueError, StopIteration) as error:
        print('분할 예시를 만들 수 없습니다:', '[details omitted from notebook output]')

In [ ]:
def fit_outcome_oof(valid, name, split_seed):
    class_mothers = valid.groupby("target")["mother_id"].nunique()
    # With 5 folds, each target needs at least max(6, 5*2)=10 distinct mothers.
    # A mother with both target values can be counted in both class totals, as originally.
    if len(class_mothers) < 2 or class_mothers.min() < max(6, CV_FOLDS * 2):
        raise ValueError("아웃컴별 양성·음성 산모가 부족합니다")
    arms = {"clinical": CLINICAL, "plus_reading": CLINICAL + READING}
    parts = {arm: [] for arm in arms}
    for fold, train, validation, held in outcome_folds(valid, split_seed):
        for arm, columns in arms.items():
            # Both arms share the same train/validation/held mothers and model seed 42.
            model = fit_catboost(train, validation, columns, SEEDS[0])
            prediction = make_prediction(held, model.predict_proba(held[columns])[:, 1])
            parts[arm].append(prediction.assign(fold=fold))
        print(name, "OOF fold", fold + 1, "complete", flush=True)
    oof = {}
    for arm, predictions in parts.items():
        prediction = pd.concat(predictions, ignore_index=True)
        # Every observed record must appear exactly once, including each twin's record.
        if prediction["row_id"].duplicated().any() or set(prediction["row_id"]) != set(valid["row_id"]):
            raise ValueError("아웃컴 OOF의 기록 범위가 불완전합니다")
        oof[arm] = prediction
    return oof

## 8. 결과별 평가와 증분을 모은다

두 모델의 OOF 예측을 모아 각각의 성능과 추가 모델 − 기준 모델의 차이를 계산한다. 같은 산모를 두 모델에서 함께 복원추출한다. 결과 하나가 실패해도 다음 결과를 분석한다. 해당 결과의 모든 fold와 두 모델의 평가가 끝나야 완료 목록에 넣는다.

In [ ]:
outcome_predictions = {}
performance_parts, paired_parts, outcome_status = [], [], []
for name, field, low, high, cutoff in OUTCOMES:
    try:
        if protocol_error or name not in outcome_data:
            raise ValueError(protocol_error or "이 결과의 관측 기록을 준비하지 못했습니다")
        oof = fit_outcome_oof(outcome_data[name], name, split_seed)
        outcome_metrics = []
        for arm, prediction in oof.items():
            metrics = metric_intervals(prediction).assign(outcome=name, arm=arm)
            outcome_metrics.append(metrics)
        # Example: AP 0.40 with reading - AP 0.35 clinical-only -> +0.05.
        difference = metric_intervals(oof["plus_reading"], oof["clinical"])
        difference = difference.assign(comparison=name + ": reading increment")
        # Publish both arms only after all folds and evaluations finish.
        outcome_predictions[name] = oof
        performance_parts.extend(outcome_metrics)
        paired_parts.append(difference)
        outcome_status.append(dict(outcome=name, status="complete"))
    except Exception as error:
        outcome_status.append(dict(outcome=name, status="unavailable", reason=str(error)))
        issues.append(dict(stage="outcome_oof", job=name, reason=str(error)))
        print(name, 'unavailable:', '[details omitted from notebook output]', flush=True)

In [ ]:
performance = pd.DataFrame(columns=METRIC_COLUMNS + ["outcome", "arm"])
outcome_paired = pd.DataFrame(columns=METRIC_COLUMNS + ["comparison"])
if performance_parts:
    performance = pd.concat(performance_parts, ignore_index=True)
if paired_parts:
    outcome_paired = pd.concat(paired_parts, ignore_index=True)
display(pd.DataFrame(outcome_status).drop(columns=["reason"], errors="ignore"))
display(performance)
# For outcome rows, the legacy "segments" column counts records, not 5-minute windows.
display(outcome_paired)

In [ ]:
if outcome_predictions:
    sample_name = next(iter(outcome_predictions))
    sample_prediction = outcome_predictions[sample_name]["plus_reading"]
    print("example outcome =", sample_name)
    print("OOF shape =", sample_prediction.shape)
    # Example: row_id "DEMO823214_0" identifies a record; fold=2 means its third held fold.
    print("records per held fold =", sample_prediction.groupby("fold").size().to_dict())

In [ ]:
plot_difference(outcome_paired, "AP", "OOF outcome prediction: added expert reading")

In [ ]:
plot_difference(outcome_paired, "AUROC", "OOF outcome prediction: added expert reading")

## 9. 이번 실행의 결과를 저장한다

원본의 파일 이름·열 순서를 유지한다. 원본의 `records.csv` 경로만 00의 준비 데이터 경로로 바꾸었다. 오류는 별도 `08_issues.csv`에도 남기며, 실패한 출력은 빈 열 구조로 갱신해 이전 실행의 성공 결과가 남지 않게 한다. 04의 Cat28 파일은 이번 실험의 입력이다.

| 파일 | 한 행 또는 내용 |
|---|---|
| `seeds/pred_with_emr_{seed}.csv` | 임상정보 추가 모델의 validation·test 구간별 점수 |
| `08_emr_same_seed.csv` | 같은 seed끼리의 시험 지표 차이, CI 없음 |
| `08_emr_paired.csv` | 선택 모델끼리의 시험 지표 차이와 CI |
| `08_outcome_cohorts.csv` | 결과별 관측·결측/범위 밖 기록과 산모 수 |
| `08_reading_association.csv` | 관측된 결과와 실제 판독 비율의 연관 |
| `outcome_{결과}_{clinical 또는 plus_reading}.csv` | 기록별 OOF 점수·fold |
| `08_outcome_metrics.csv`, `08_outcome_paired.csv` | OOF 성능·증분과 산모 CI |
| `08_status.json`, `08_issues.csv` | 완료 상태·seed 선택·실패 이유 |

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "seeds").mkdir(exist_ok=True)
for seed in SEEDS:
    prediction = emr_predictions.get(seed, pd.DataFrame(columns=PREDICTION_COLUMNS))
    prediction.to_csv(RESULTS / "seeds" / f"pred_with_emr_{seed}.csv", index=False)
emr_paired.to_csv(RESULTS / "08_emr_paired.csv", index=False)
emr_same_seed.to_csv(RESULTS / "08_emr_same_seed.csv", index=False)
cohorts.to_csv(RESULTS / "08_outcome_cohorts.csv", index=False)
associations.to_csv(RESULTS / "08_reading_association.csv", index=False)

for name, field, low, high, cutoff in OUTCOMES:
    for arm in ("clinical", "plus_reading"):
        # A failed outcome writes an empty file for BOTH arms, replacing stale OOF rows.
        prediction = outcome_predictions.get(name, {}).get(arm)
        if prediction is None:
            prediction = pd.DataFrame(columns=OUTCOME_PREDICTION_COLUMNS)
        prediction.to_csv(RESULTS / f"outcome_{name}_{arm}.csv", index=False)
performance.to_csv(RESULTS / "08_outcome_metrics.csv", index=False)
outcome_paired.to_csv(RESULTS / "08_outcome_paired.csv", index=False)

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
issue_table.to_csv(RESULTS / "08_issues.csv", index=False)
# Keep the original completion fields and add explicit model/selection/error details.
status = dict(
    outcomes=outcome_status, emr_completed_seeds=len(same_seed_parts),
    emr_fitted_seeds=sorted(emr_predictions), emr_selected_seed=selected_emr_seed,
    emr_paired_complete=not emr_paired.empty, emr_selection=emr_candidates,
    protocol_error=protocol_error, records_error=records_error, emr_input_error=emr_input_error,
)
(RESULTS / "08_status.json").write_text(
    json.dumps(status, ensure_ascii=False, indent=2), encoding="utf-8",
)
print('saved to =', RESULTS)
print("EMR paired complete =", status["emr_paired_complete"])
print("outcomes complete =", len(outcome_predictions), "/", len(OUTCOMES))
if not issue_table.empty:
    print("issue count =", len(issue_table))

## 해석의 범위

첫 분석은 전문가 판독을 예측하는 데 임상정보가 더하는 값을 본다. 재태주수가 실제 모니터링 시점에 가용했는지는 별도 확인이 필요하다.

두 번째 분석은 관측된 분만 결과와 실제 전문가 판독 요약의 탐색적 연관·예측 비교다. 모델이 예측한 판독 점수를 분만 결과 입력으로 넣은 실험은 아니다. 결과 관측의 선택 편향과 모니터링부터 분만까지의 시간은 해결되지 않는다.

OOF 신뢰구간은 고정된 예측을 산모 단위로 재표집한 평가 표본의 불확실성이다. 모델을 다시 학습하거나 fold를 새로 나누는 변동을 포함하지 않는다. CI가 0을 포함한다는 이유만으로 판독 정보가 없다고 결론내리지 않는다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])